In [1]:
import csv
import glob
import os
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import glob
import pandas as pd
import numpy as np
import torch
from collections import defaultdict

CSV_GLOB = "/project2/ll_774_951/uk_ru/twitter/data/*/*.csv"

COLS = ['userid', 'hashtag', 'rt_hashtag', 'text', 'location', 'description', 'urls_list', 'rt_urls_list', 'date']

pd.set_option('display.max_columns', None)

In [2]:
files = sorted(glob.glob(CSV_GLOB))
print(f"Found {len(files)} CSV files")

chunks = []
skipped = 0
for i, f in enumerate(files[:10]):
    try:
        print("(i) reading ", f, flush=True, end='\r')
        temp = pd.read_csv(f,
                              # usecols=COLS, 
                              engine='python', on_bad_lines='skip')
        temp['filename'] = f
        chunks.append(temp)
    except Exception as e:
        print(f"Skipping {f}: {e}")
        skipped += 1
df = pd.concat(chunks, ignore_index=True)
df['userid'] = pd.to_numeric(df['userid'], errors='coerce')
df = df.dropna(subset=['userid']).copy()
df['userid'] = df['userid'].astype(int)
print(f"Skipped files: {skipped}")
print(f"Total rows: {len(df):,}  |  Unique users: {df['userid'].nunique():,}")

Found 1498 CSV files
Skipped files: 0oject2/ll_774_951/uk_ru/twitter/data/2022-02/ukraine_russia-2022-02-22-13.csv
Total rows: 1,575,480  |  Unique users: 637,762


## mappers

In [3]:
domain2score = {
    'abcnews.go.com': 2.0,
     'bbc.com': 3.0,
     'breitbart.com': 5.0,
     'bostonglobe.com': 2.0,
     'businessinsider.com': 3.0,
     'buzzfeednews.com': 1.0,
     'cbsnews.com': 2.0,
     'chicagotribune.com': 3.0,
     'cnbc.com': 3.0,
     'cnn.com': 2.0,
     'dailycaller.com': 5.0,
     'dailymail.co.uk': 5.0,
     'foxnews.com': 4.0,
     'huffpost.com': 1.0,
     'infowars.com': 5.0,
     'latimes.com': 2.0,
     'msnbc.om': 1.0,
     'nbcnews.com': 2.0,
     'nytimes.com': 2.0,
     'npr.org': 3.0,
     'oann.com': 4.0,
     'pbs.org': 3.0,
     'reuters.com': 3.0,
     'theguardian.com': 2.0,
     'usatoday.com': 3.0,
     'yahoo.com': 2.0,
     'vice.com': 1.0,
     'washingtonpost.com': 2.0,
     'wsj.com': 3.0,
      
    # additional
     'thehill.com': 2.7,
     'rt.com': 3.7,
     'rawstory.com': 1.7,
     'news.sky.com': 2.3,
     'independent.co.uk': 2.3,
     'dailykos.com': 1.7
}

In [4]:
domain2canonical = {
    
    'bit.ly': None,  # URL shortener
    'dlvr.it': None,  # syndication tool
    'trib.al': None,  # URL shortener
    'ift.tt': None,  # IFTTT shortener
    'twibbon.com': None,  # campaign tool
    't.me': None,  # Telegram shortener
    'apple.news': None,  # aggregator
    'ow.ly': None,  # Hootsuite shortener
    'buff.ly': None,  # Buffer shortener
    'tinyurl.com': None,  # URL shortener
    'news.google.com': None,  # aggregator
    'lnr.app': None,  # URL shortener
    'fiverr.com': None,  # freelancing platform, likely data artifact
    
    'twitter.com': 'twitter.com',
    'reut.rs': 'reuters.com',
    'youtu.be': 'youtube.com',
    'theguardian.com': 'theguardian.com',
    'youtube.com': 'youtube.com',
    'nytimes.com': 'nytimes.com',
    'reuters.com': 'reuters.com',
    'mfa.gov.tr': 'mfa.gov.tr',
    'washingtonpost.com': 'washingtonpost.com',
    'cnn.com': 'cnn.com',
    'businessinsider.com': 'businessinsider.com',
    'rt.com': 'rt.com',
    'hill.cm': 'thehill.com',
    'bbc.in': 'bbc.com',
    'foxnews.com': 'foxnews.com',
    'facebook.com': 'facebook.com',
    'rawstory.com': 'rawstory.com',
    'bbc.com': 'bbc.com',
    'bbc.co.uk': 'bbc.com',
    'news.sky.com': 'news.sky.com',
    'melenchon.fr': 'melenchon.fr',
    'a.msn.com': 'msn.com',
    'npr.org': 'npr.org',
    'politico.com': 'politico.com',
    'en.wikipedia.org': 'wikipedia.org',
    'cnb.cx': 'cnbc.com',
    'timesofindia.indiatimes.com': 'timesofindia.indiatimes.com',
    'dailykos.com': 'dailykos.com',
    'independent.co.uk': 'independent.co.uk',
    'bfmtv.com': 'bfmtv.com',
    'msn.com': 'msn.com',
    'jp.reuters.com': 'reuters.com',
    'axios.com': 'axios.com',
    'babylonbee.com': 'babylonbee.com',
    'news.yahoo.com': 'yahoo.com',
    'gelecektenhaber.com': 'gelecektenhaber.com',
    'instagram.com': 'instagram.com',
    'google.com': 'google.com',
    'zerohedge.com': 'zerohedge.com',
    'tagesschau.de': 'tagesschau.de',
    'thehill.com': 'thehill.com',
    'cnbc.com': 'cnbc.com',
    'bloomberg.com': 'bloomberg.com',
    'lemonde.fr': 'lemonde.fr',
    'ndtv.com': 'ndtv.com',
    'forbes.com': 'forbes.com',
    'ti.me': 'time.com',
    'aninews.in': 'aninews.in',
    'wsj.com': 'wsj.com',
    'theatlantic.com': 'theatlantic.com',
    'spiegel.de': 'spiegel.de',
    'on.ft.com': 'ft.com',
    'en.kremlin.ru': 'kremlin.ru',
    'cnn.it': 'cnn.com',
    'whitehouse.gov': 'whitehouse.gov',
    'insiderpaper.com': 'insiderpaper.com',
    'abc.net.au': 'abc.net.au',
    'francetvinfo.fr': 'francetvinfo.fr',
    'edition.cnn.com': 'cnn.com',
    'on.rt.com': 'rt.com',
    'aljazeera.com': 'aljazeera.com',
    'mobile.twitter.com': 'twitter.com',
    'parafesor.net': 'parafesor.net',
    'who.int': 'who.int',
    'world.bistosh.com': 'bistosh.com',
    'es.rt.com': 'rt.com',
    'coinkurry.com': 'coinkurry.com',
    'presidentti.fi': 'presidentti.fi',
    'nos.nl': 'nos.nl',
    'sputniknews.com': 'sputniknews.com',
    'mol.im': 'dailymail.co.uk',
    'vilaweb.cat': 'vilaweb.cat',
    'bild.de': 'bild.de',
    'washingtontimes.com': 'washingtontimes.com',
    'scmp.com': 'scmp.com',
    'wapo.st': 'washingtonpost.com',
    'aajtak.in': 'aajtak.in',
    'nbcnews.com': 'nbcnews.com',
    'tf1info.fr': 'tf1info.fr',
    'rumble.com': 'rumble.com',
    'maisinwin.blogspot.com': 'maisinwin.blogspot.com',
    'welt.de': 'welt.de',
    'nypost.com': 'nypost.com',
    'ft.com': 'ft.com',
    'liveuamap.com': 'liveuamap.com',
    'moneycontrol.com': 'moneycontrol.com',
    'kremlin.ru': 'kremlin.ru',
    
}

In [5]:
handle2score = {
    'abc': 2,
 'bbcworld': 3,
 'breitbartnews': 5,
 'bostonglobe': 2,
 'businessinsider': 3,
 'buzzfeednews': 1,
 'cbsnews': 2,
 'chicagotribune': 3,
 'cnbc': 3,
 'cnn': 2,
 'dailycaller': 5,
 'dailymail': 5,
 'foxnews': 4,
 'huffpost': 1,
 'infowars*': 5,
 'latimes': 2,
 'msnbc': 1,
 'nbcnews': 2,
 'nytimes': 2,
 'npr': 3,
 'oann': 4,
 'pbs': 3,
 'reuters': 3,
 'guardian': 2,
 'usatoday': 3,
 'yahoonews': 2,
 'vice': 1,
 'washingtonpost': 2,
 'wsj': 3,
 # additional
 'thehill': 2.7,
 'rt_com': 3.7,
 'rawstory': 1.7,
 'skynews': 2.3,
 'independent': 2.3,
 'dailykos': 1.7
}

In [6]:
hashtag2score = {'blm': 0,'resist': 0,'fbpe': 0,'blacklivesmatter': 0,'fbr': 0,'maga': 1,'theresistance': 0,'voteblue': 0,'resistance': 0,'bidenharris': 0,'johnsonout': 0,'lgbtq': 0,'bidenharris2020': 0,'2a': 1,'fbpa': 0,'resister': 0,'fbppr': 0,'bluewave': 0,'gtto': 0,'freepalestine': 0,'rejoineu': 0,'voteblue2022': 0,'kag': 1,'wearamask': 0,'getvaccinated': 0,'humanrights': 0,'votingrights': 0,'science': 0,'goodtrouble': 0,'strongertogether': 0,'stillwithher': 0,'climatecrisis': 0,'metoo': 0,'demvoice1': 0,'biden': 0,'climatechange': 0,'justicematters': 0,'americafirst': 1,'nevertrump': 0,'khive': 0,'democrat': 0,'vaccinated': 0,'buildbackbetter': 0,'stopasianhate': 0,'prochoice': 0,'drcole': 1,'bds': 0,'votebluenomatterwho': 0,'teampelosi': 0,'handmarkedpaperballots': 1,'reconquête': 1,'biden2020': 0,'patriot': 1,'equality': 0,'prolife': 1,'antifa': 0,'fjb': 1,'lgbt': 0,'nra': 1,'climateaction': 0,'unionpopulaire': 0,'zemmour2022': 1,'trump': 1,'demcast': 0,'m4a': 0,'vaxxed': 0,'democracy': 0,'indivisible': 0,'teamjustice': 0,'noafd': 0,'alllivesmatter': 1}

In [29]:
for col in ['urls_list', 'rt_urls_list', 'qtd_urls_list']:
    i = df[col].str.len().gt(2)
    df[col+"_"] = df.loc[i, col].str.findall("display_url': '([^/]+)/")
    
## URL
# domain2canonical = {'en.wikipedia.org': 'wikipedia.org', ...}
# domain2score = {'abcnews.go.com': 2.0,'bbc.com': 3.0, ...}
url_scores = (
    df['urls_list_'].dropna()
    .apply(lambda x: [domain2canonical.get(u, u) for u in x])
    .apply(lambda x: [domain2score.get(u) for u in x if u in domain2score])
)
url_scores = url_scores[url_scores.str.len().gt(0)]
df['url_scores'] = url_scores
mask = df.url_scores.notna()
df['mean_tweet_url_pol_score'] = np.nan
mean_tweet_url_pol_score = df.loc[mask, 'url_scores'].apply(
    lambda x: np.nanmean([0 if y < 3 else 1 if y > 3 else np.nan for y in x])
)
mean_tweet_url_pol_score = mean_tweet_url_pol_score[~mean_tweet_url_pol_score.between(1/5,4/5)]
df['mean_tweet_url_pol_score'] = mean_tweet_url_pol_score
user2urlScore = pd.crosstab(df['userid'], df['mean_tweet_url_pol_score'])
user2urlScore.columns = ['url_left', 'url_right']

## RETWEET
# handle2score = {'abc': 2, 'bbcworld': 3,...}
rt_org_pol_score = df.rt_screen.str.lower().map(handle2score)
df['rt_org_pol_score'] = np.nan
i = rt_org_pol_score.gt(3)
df.loc[i, 'rt_org_pol_score'] = 'rt_right'
j = rt_org_pol_score.lt(3)
df.loc[j, 'rt_org_pol_score'] = 'rt_left'
user2rtScore = pd.crosstab(df['userid'], df['rt_org_pol_score'])

## HASHTAG
# hashtag2score = {'blm': 0,'resist': 0,'fbpe': 0, ...}
df["descr_hashtags"] = df.drop_duplicates(subset=['userid', 'description']).description.str.lower().str.findall(r'\#(\w+)')
i = df.descr_hashtags.str.len().lt(1)
df.loc[i, 'descr_hashtags'] = np.nan
hashtag_scores = (
    df['descr_hashtags'].dropna()
    .apply(lambda x: [hashtag2score.get(u) for u in x if u in hashtag2score])
)
hashtag_scores = hashtag_scores[hashtag_scores.str.len().gt(0)]
hashtag_scores = hashtag_scores.apply(lambda x: pd.Series(x).value_counts())
df[['hashtag_left', 'hashtag_right']] = hashtag_scores
user2hashtagScore = df.groupby('userid')[['hashtag_left', 'hashtag_right']].sum()

## USER2SCORE
user2score = pd.concat([user2urlScore, user2rtScore, user2hashtagScore], axis=1, join='outer')
left_cols = ['url_left', 'rt_left', 'hashtag_left']
right_cols = ['url_right', 'rt_right', 'hashtag_right']
user2score['left_vote'] = user2score[left_cols].sum(1)
user2score['right_vote'] = user2score[right_cols].sum(1)

user2pol = user2score.right_vote.gt(0) & user2score.left_vote.gt(0)
user2pol = user2pol * 0 - 1
right_mask = user2score.right_vote.gt(1) & user2score.left_vote.eq(0)
user2pol[right_mask] = 1
left_mask = user2score.right_vote.eq(0) & user2score.left_vote.gt(1)
user2pol[left_mask] = 0
user2pol.value_counts()

/tmp/SLURM_7835654/ipykernel_688853/2811079227.py:18: RuntimeWarning: Mean of empty slice
  lambda x: np.nanmean([0 if y < 3 else 1 if y > 3 else np.nan for y in x])
/tmp/SLURM_7835654/ipykernel_688853/2811079227.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rt_right' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[i, 'rt_org_pol_score'] = 'rt_right'


-1    627289
 0      8697
 1      1776
Name: count, dtype: int64

In [53]:
d = user2score[['left_vote', 'right_vote']].gt(1)
d[d.sum(1).gt(1)] = False

In [56]:
d.value_counts(normalize=True)

userid
3839                  -1
12301                  0
13621                 -1
428333                 0
620233                -1
                      ..
1496119850690912256   -1
1496119911554371584   -1
1496121207246237696   -1
1496121427212357632   -1
1496121536830480384   -1
Length: 637762, dtype: int64